# 8-4절 연습 문제 풀이

이 노트북은 8-4절 연습 문제(8-12 ~ 8-16)의 풀이 예시다.

- 본문 예제 코드는 `code_examples/ch08/08-04_example.ipynb`를 참고한다.
- 흉부 X선 데이터셋은 본문 예제와 같은 경로를 사용한다.
- 문제마다 **풀이 해설**과 **문제 검토**를 함께 실었다. 문제 검토는 최종 검토 3단계의 기록이다.

## 공통 준비 — 환경 설정, 데이터, 학습 함수

In [1]:
# 환경 설정
import sys
sys.path.append('../../')

from code_reference import common
from code_reference import visualize as viz

common.set_korean_plot_env()
viz.configure(save_grayscale=False)

SEED = 42
common.set_seed(SEED, deterministic=True)
device = common.get_device()

CUDA를 사용합니다.


In [2]:
# 흉부 X선 데이터셋 경로 확인 (본문 예제에서 이미 내려받은 것을 사용)
from pathlib import Path

xray_root = Path('../../download/kaggle/chest_xray/chest_xray')
if not (xray_root / 'train').exists():
    raise FileNotFoundError(
        f'{xray_root}에 train 디렉터리가 없다. '
        '08-04_example.ipynb의 다운로드 셀을 먼저 실행하자.')

for split in ('train', 'val', 'test'):
    counts = {c.name: len(list(c.glob('*'))) for c in sorted((xray_root / split).iterdir())
              if c.is_dir()}
    print(f'{split:6s} {counts}  합계 {sum(counts.values())}')

train  {'NORMAL': 1341, 'PNEUMONIA': 3875}  합계 5216
val    {'NORMAL': 8, 'PNEUMONIA': 8}  합계 16
test   {'NORMAL': 234, 'PNEUMONIA': 390}  합계 624


In [3]:
# 전이 학습에 사용하는 함수 (본문 예제와 같은 방식)
import copy

import timm
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision.datasets import ImageFolder

BATCH_SIZE = 32
EPOCHS = 20
PATIENCE = 5
LR = 1e-3


@torch.no_grad()
def get_accuracy(model, data_loader, device):
    model.eval()
    correct_size, sample_size = 0, 0
    for inputs, labels in data_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        outputs = model(inputs)
        correct_size += (outputs.argmax(dim=1) == labels).sum().item()
        sample_size += inputs.size(0)
    return correct_size / sample_size


@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    loss_sum, correct_size, sample_size = 0.0, 0, 0
    for inputs, labels in loader:
        inputs, labels = inputs.to(device), labels.to(device)
        outputs = model(inputs)
        loss_sum += criterion(outputs, labels).item() * inputs.size(0)
        correct_size += (outputs.argmax(dim=1) == labels).sum().item()
        sample_size += inputs.size(0)
    return loss_sum / sample_size, correct_size / sample_size


def transfer_train(model, train_loader, valid_loader, test_loader, name,
                   epochs=EPOCHS, patience=PATIENCE, lr=LR,
                   param_groups=None):
    """param_groups를 주면 [코드 7-9]처럼 계층별로 다른 학습률을 적용한다"""
    model.to(device)
    criterion = nn.CrossEntropyLoss()
    params = [p for p in model.parameters() if p.requires_grad]
    optimizer = (optim.Adam(param_groups) if param_groups
                 else optim.Adam(params, lr=lr))
    log = common.EpochLogger(epochs, target_rows=epochs)
    best_valid_loss, best_epoch, best_params, patience_counter = float('inf'), -1, None, 0
    print(f'{name} 학습 (학습 대상 파라미터 {sum(p.numel() for p in params):,}개)')
    for epoch in range(1, epochs + 1):
        model.train()
        loss_sum, sample_size = 0.0, 0
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()
            loss = criterion(model(inputs), labels)
            loss.backward()
            optimizer.step()
            loss_sum += loss.item() * inputs.size(0)
            sample_size += inputs.size(0)
        valid_loss, valid_acc = evaluate(model, valid_loader, criterion, device)
        log.row(epoch, loss_sum / sample_size, valid_loss, valid_acc * 100)
        if valid_loss < best_valid_loss:
            best_valid_loss, best_epoch = valid_loss, epoch
            best_params = copy.deepcopy(model.state_dict())
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= patience:
                break
    if best_params is not None:
        print(f'최적 에포크({best_epoch}, 최소 검증 손실 {best_valid_loss:.4f})의 파라미터로 복원')
        model.load_state_dict(best_params)
    test_acc = get_accuracy(model, test_loader, device)
    print(f'{name}: 최적 에포크 {best_epoch}, 평가 정확도 {test_acc * 100:.2f}%')
    return model, {'name': name, 'best_epoch': best_epoch,
                   'test_acc': test_acc * 100}


results = {}

/home/crapas/.local/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


---

## 연습 문제 8-12

> [코드 8-7]에서 VGG-16 대신 ResNet-18을 사용해 이미지를 분류해 보자.

In [4]:
# [코드 8-7]의 vgg16_bn을 resnet18로 바꾸기만 하면 된다
import requests
from PIL import Image

data_root = '../../data'
img = Image.open(data_root + '/cat.jpg')

url = ('https://raw.githubusercontent.com/pytorch/hub/master/'
       'imagenet_classes.txt')
labels = requests.get(url, timeout=30).text.splitlines()


def classify_top5(model_name, image):
    """timm 모델 이름을 받아 상위 5개 클래스를 출력한다"""
    model = timm.create_model(model_name, pretrained=True).to(device)
    # 모델마다 입력 크기와 정규화 값이 다르므로 설정에서 변환 객체를 만든다
    transform_config = timm.data.resolve_data_config(model.default_cfg)
    transform = timm.data.create_transform(**transform_config)
    input_tensor = transform(image).unsqueeze(0).to(device)
    model.eval()
    with torch.no_grad():
        output = model(input_tensor)
    scores, top5_idx = torch.topk(output.softmax(dim=1), 5)
    print(f'[{model_name}] 입력 크기 {transform_config["input_size"]}')
    for i in range(5):
        print(f'  {i + 1}위: {labels[top5_idx[0, i].item()]} '
              f'({scores[0, i].item() * 100:.2f}%)')
    return [labels[top5_idx[0, i].item()] for i in range(5)]


top5_vgg = classify_top5('vgg16_bn', img)
print()
top5_resnet = classify_top5('resnet18', img)

[vgg16_bn] 입력 크기 (3, 224, 224)
  1위: tiger cat (50.27%)
  2위: Egyptian cat (24.83%)
  3위: tabby (12.99%)
  4위: lynx (3.15%)
  5위: tiger (3.05%)



[resnet18] 입력 크기 (3, 224, 224)
  1위: tiger cat (60.60%)
  2위: Egyptian cat (12.02%)
  3위: tabby (9.56%)
  4위: lynx (3.50%)
  5위: weasel (0.90%)


In [5]:
# 두 모델의 결과 비교
print(f"{'순위':<6}{'VGG-16':<22}{'ResNet-18':<22}")
print('-' * 50)
for i, (a, b) in enumerate(zip(top5_vgg, top5_resnet), start=1):
    print(f'{i:<6}{a:<22}{b:<22}')
print()
common_labels = set(top5_vgg) & set(top5_resnet)
print(f'두 모델의 상위 5개에 공통으로 든 클래스: {len(common_labels)}개')
print(f'  {sorted(common_labels)}')

순위    VGG-16                ResNet-18             
--------------------------------------------------
1     tiger cat             tiger cat             
2     Egyptian cat          Egyptian cat          
3     tabby                 tabby                 
4     lynx                  lynx                  
5     tiger                 weasel                

두 모델의 상위 5개에 공통으로 든 클래스: 4개
  ['Egyptian cat', 'lynx', 'tabby', 'tiger cat']


### 풀이 해설 — 연습 문제 8-12

[코드 8-7]에서 `timm.create_model('vgg16_bn', ...)`을 `timm.create_model('resnet18', ...)`으로
바꾸기만 하면 된다. **한 줄만 고치면 되는 것이 이 문제의 요지다.**

여기서 눈여겨볼 것은 데이터 변환 객체를 모델에서 끌어낸다는 점이다.

```python
transform_config = timm.data.resolve_data_config(model.default_cfg)
transform = timm.data.create_transform(**transform_config)
```

모델마다 학습할 때 쓴 입력 크기와 정규화 값이 다른데, 이 두 줄이 그 차이를 알아서 흡수한다.
만약 VGG-16용 변환 객체를 그대로 재사용했다면 모델이 기대하는 전처리와 어긋나 엉뚱한 결과가
나올 수 있다. 8-4절 본문이 "timm이 단순히 모델만 제공하지 않고, 활용에 필요한 여러 기능도 함께
제공한다"고 말한 대목이 실제로 쓰이는 자리다.

두 모델의 예측을 견주면 구조가 달라도 비슷한 클래스를 고른다는 것을 확인할 수 있다. 실행
결과는 위 표로 확인한다.

---

## 연습 문제 8-13

> [코드 8-9]는 ResNet-50의 출력 크기가 1,000인 출력층 선형 계층을 출력 크기가 2인 선형 계층으로
> 교체한다. 교체하는 대신 출력층은 그대로 두고 그 뒤에 입력 크기가 1,000, 출력 크기가 2인 선형
> 계층을 덧붙이면 전이 학습 성능은 어떻게 달라질까? 직접 모델을 만들어 확인하고, 왜 그런 결과가
> 나오는지 해석해 보자.

In [6]:
# 데이터셋과 데이터로더 (본문 예제와 동일)
base_model = timm.create_model('resnet50', pretrained=True)
transform_config = timm.data.resolve_data_config(base_model.default_cfg)
transform = timm.data.create_transform(**transform_config)

train_set = ImageFolder(root=xray_root / 'train', transform=transform)
valid_set = ImageFolder(root=xray_root / 'val', transform=transform)
test_set = ImageFolder(root=xray_root / 'test', transform=transform)

train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True)
valid_loader = DataLoader(valid_set, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_set, batch_size=BATCH_SIZE, shuffle=False)

print(f'클래스: {train_set.classes}')
print(f'훈련 {len(train_set)}, 검증 {len(valid_set)}, 평가 {len(test_set)}')

클래스: ['NORMAL', 'PNEUMONIA']
훈련 5216, 검증 16, 평가 624


In [7]:
# (기준) [코드 8-9]와 같은 방식 - 출력층을 교체
def build_replaced():
    common.set_seed(SEED, deterministic=True)
    model = timm.create_model('resnet50', pretrained=True)
    for param in model.parameters():
        param.requires_grad = False
    model.fc = nn.Linear(2048, 2)
    return model


# (비교) 출력층을 그대로 두고 그 뒤에 선형 계층을 덧붙이는 방식
def build_appended():
    common.set_seed(SEED, deterministic=True)
    model = timm.create_model('resnet50', pretrained=True)
    for param in model.parameters():
        param.requires_grad = False
    # 기존 fc(2048 -> 1000)는 그대로 두고, 그 뒤에 1000 -> 2 계층을 잇는다
    model.fc = nn.Sequential(
        model.fc,               # 사전 학습된 출력층(고정)
        nn.Linear(1000, 2)      # 새로 더한 선형 계층(학습 대상)
    )
    return model


for factory, label in ((build_replaced, '교체 방식'), (build_appended, '추가 방식')):
    m = factory()
    trainable = sum(p.numel() for p in m.parameters() if p.requires_grad)
    print(f'{label}: 학습 대상 파라미터 {trainable:,}개')

교체 방식: 학습 대상 파라미터 4,098개


추가 방식: 학습 대상 파라미터 2,002개


In [8]:
model_replaced, results['replaced'] = transfer_train(
    build_replaced(), train_loader, valid_loader, test_loader, '교체 방식(특징 추출)')

교체 방식(특징 추출) 학습 (학습 대상 파라미터 4,098개)


에포크    훈련 손실    검증 손실    정확도(%)     시간
  1/20       0.3651       0.4780       75.00%     1:19


  2/20       0.2287       0.4042       75.00%     2:36


  3/20       0.1923       0.3777       75.00%     3:53


  4/20       0.1790       0.3361       81.25%     5:12


...

  8/20       0.1448       0.3912       75.00%    10:34


  9/20       0.1319       0.3122       81.25%    11:50


 10/20       0.1376       0.3687       75.00%    13:06


 11/20       0.1297       0.3274       75.00%    14:23
최적 에포크(6, 최소 검증 손실 0.3074)의 파라미터로 복원


교체 방식(특징 추출): 최적 에포크 6, 평가 정확도 80.29%


In [9]:
model_appended, results['appended'] = transfer_train(
    build_appended(), train_loader, valid_loader, test_loader, '추가 방식(특징 추출)')

추가 방식(특징 추출) 학습 (학습 대상 파라미터 2,002개)


에포크    훈련 손실    검증 손실    정확도(%)     시간
  1/20       0.7885       0.3590       81.25%     1:21


  2/20       0.4809       0.4442       81.25%     2:41


  3/20       0.4074       0.2383       93.75%     4:02


  4/20       0.5466       0.3360       87.50%     5:17


...

 14/20       0.3661       0.1800       93.75%    17:58


 15/20       0.3900       0.3042       81.25%    19:12


 16/20       0.3137       0.1468       93.75%    20:26


 17/20       0.3062       0.4987       68.75%    21:40
최적 에포크(12, 최소 검증 손실 0.1373)의 파라미터로 복원


추가 방식(특징 추출): 최적 에포크 12, 평가 정확도 78.85%


In [10]:
print(f"{'방식':<26}{'최적 에포크':>10}{'평가 정확도':>14}")
print('-' * 50)
for k in ('replaced', 'appended'):
    r = results[k]
    print(f"{r['name']:<26}{r['best_epoch']:>10}{r['test_acc']:>13.2f}%")

방식                            최적 에포크        평가 정확도
--------------------------------------------------
교체 방식(특징 추출)                       6        80.29%
추가 방식(특징 추출)                      12        78.85%


### 풀이 해설 — 연습 문제 8-13

두 방식의 차이는 **분류기가 무엇을 입력으로 받는가**에 있다.

- **교체 방식**: 전역 평균 풀링이 뽑아낸 **2,048차원 특징 벡터**를 곧바로 받는다. 이 벡터는
  ImageNet 분류를 위해 학습된 것이지만, 특정 클래스에 맞춰 압축되기 전의 **일반적인 시각 특징**을
  담고 있다.
- **추가 방식**: 사전 학습된 `fc`를 거친 **1,000차원 로짓**을 받는다. 이 값은 "이 이미지가
  ImageNet의 1,000개 클래스 각각일 점수"로, **ImageNet의 클래스 체계에 맞춰 이미 해석이 끝난**
  결과다.

흉부 X선 사진에는 ImageNet의 1,000개 클래스(개, 고양이, 자동차 등)에 해당하는 것이 없다.
그래서 1,000차원 로짓은 X선 사진을 구분하는 데 필요한 정보를 거의 담지 못한다. 2,048차원에서
1,000차원으로 줄어드는 과정에서 **폐렴 판독에 쓸 수 있었던 일반적인 특징이 버려지는** 셈이다.

또 하나. 새로 더한 계층이 사전 학습된 `fc`와 곧바로 이어지는데, 그 사이에 활성화 함수가 없다.
`fc`가 고정되어 있으므로 두 선형 계층은 하나의 선형 변환과 다를 바 없어, 표현력도 늘지 않는다.

**정리하면 특징 추출 방식의 전이 학습에서는 "어느 계층의 출력을 특징으로 삼을 것인가"가 성능을
가른다.** 이 문제는 그 선택이 왜 중요한지를 극단적인 예로 보여 준다. 실행 결과는 위 표로
확인한다.

---

## 연습 문제 8-14

> 흉부 X선 분류 모델을 만들 때 사전 학습된 ResNet-50을 특징 추출 대신 미세 조정 방식으로
> 사용하도록 예제를 수정해 보자.

In [11]:
# 미세 조정 방식: 파라미터를 고정하지 않고 모델 전체를 학습한다
def build_finetune():
    common.set_seed(SEED, deterministic=True)
    model = timm.create_model('resnet50', pretrained=True)
    # 파라미터 고정 과정이 없으므로 출력층 교체 순서를 따질 필요가 없다
    model.fc = nn.Linear(2048, 2)
    return model


m = build_finetune()
print(f'미세 조정: 학습 대상 파라미터 '
      f'{sum(p.numel() for p in m.parameters() if p.requires_grad):,}개')
print(f'특징 추출: 학습 대상 파라미터 '
      f'{sum(p.numel() for p in build_replaced().parameters() if p.requires_grad):,}개')

미세 조정: 학습 대상 파라미터 23,512,130개


특징 추출: 학습 대상 파라미터 4,098개


In [12]:
# 미세 조정의 학습률 - [코드 7-9]에서 배운 계층별 학습률을 적용한다
#     사전 학습된 계층은 매우 작은 학습률로 살짝만 움직이고,
#     새로 교체한 출력층은 평소 학습률로 빠르게 배운다
LR_FT = 1e-5        # 사전 학습된 계층에 적용할 학습률
model_ft = build_finetune()

finetune_groups = [
    # 사전 학습된 계층의 파라미터는 낮은 학습률 지정
    {'params': [p for name, p in model_ft.named_parameters()
                if not name.startswith('fc.')], 'lr': LR_FT},
    # 새로 교체한 출력층의 파라미터는 통상 학습률 지정
    {'params': model_ft.fc.parameters(), 'lr': LR},
]

model_finetune, results['finetune'] = transfer_train(
    model_ft, train_loader, valid_loader, test_loader,
    '미세 조정', param_groups=finetune_groups)


미세 조정 학습 (학습 대상 파라미터 23,512,130개)


에포크    훈련 손실    검증 손실    정확도(%)     시간
  1/20       0.3255       0.6868       68.75%     1:24


  2/20       0.1741       0.5683       68.75%     2:46


  3/20       0.1377       0.2979       81.25%     4:07


  4/20       0.1205       0.3336       75.00%     5:27


  5/20       0.1004       0.4141       87.50%     6:47


  6/20       0.0857       0.3874       87.50%     8:07


  7/20       0.0715       0.4001       75.00%     9:25


  8/20       0.0596       0.6580       68.75%    10:44
최적 에포크(3, 최소 검증 손실 0.2979)의 파라미터로 복원


미세 조정: 최적 에포크 3, 평가 정확도 82.05%


In [13]:
print(f"{'방식':<26}{'최적 에포크':>10}{'평가 정확도':>14}")
print('-' * 50)
for k in ('replaced', 'finetune'):
    r = results[k]
    print(f"{r['name']:<26}{r['best_epoch']:>10}{r['test_acc']:>13.2f}%")

방식                            최적 에포크        평가 정확도
--------------------------------------------------
교체 방식(특징 추출)                       6        80.29%
미세 조정                              3        82.05%


### 풀이 해설 — 연습 문제 8-14

코드 수정은 간단하다. [코드 8-9]에서 **파라미터를 고정하는 반복문만 지우면 된다.** 본문이
"미세 조정 방식도 같은 방법으로 출력층을 교체한다. 다만 모든 파라미터를 학습하므로 교체와
고정의 순서를 따질 필요가 없다"고 밝힌 그대로다.

다만 코드에 드러나지 않는 조정이 하나 더 있다. **학습률이다.** 미세 조정은 이미 잘 학습된
파라미터를 출발점으로 삼으므로, 특징 추출과 같은 학습률(1e-3)을 쓰면 첫 몇 배치에서 사전
학습된 값이 크게 흔들려 오히려 성능이 떨어진다.

이 문제는 7장이 이미 다뤘다. [코드 7-9]가 **파라미터 그룹별로 다른 학습률을 주는 방법**을
보여 주며, "사전 학습 계층의 파라미터는 매우 작은 값(보통 1e-5 수준)으로, 추가한 계층의
파라미터는 평소 수준의 값(보통 1e-3 수준)으로 다르게 적용한다"고 정리했다. 위 셀은 그
방식을 그대로 가져와, 사전 학습된 계층에는 `LR_FT = 1e-5`를, 새로 교체한 `fc`에는 `LR = 1e-3`을
주었다.

8-4절 본문은 미세 조정을 소개하면서 학습률 이야기를 하지 않는다. 7장을 기억하는 독자라면
스스로 [코드 7-9]를 떠올리겠지만, 그렇지 않으면 단일 학습률로 풀다가 성능이 떨어지는 결과를
얻고 원인을 찾지 못한다. 아래 문제 검토에 이 점을 적었다.

미세 조정이 성능을 끌어올리는 이유는 **합성곱 계층까지 흉부 X선 사진에 맞춰 다시 학습되기**
때문이다. ImageNet으로 익힌 특징은 자연 이미지의 경계와 질감에 맞춰져 있어, 회색조 X선 사진의
폐 음영 패턴과는 결이 다르다. 미세 조정은 그 간극을 메운다. 8-4절 본문이 성능이 낮은 이유로
"미세 조정이 아닌 특징 추출 방식을 사용했다는 점"을 첫 번째로 든 것과 이어진다.


---

## 연습 문제 8-15 [도전 문제]

> 8-4절은 공개 데이터셋을 받은 그대로 사용했다. 포함된 훈련, 검증, 평가 데이터셋의 샘플 수와
> 클래스별 구성을 확인하고 다음 질문에 답해 보자.
> - 이 검증 데이터셋으로 조기 종료 시점을 판단할 때 어떤 문제가 생길 수 있을까?
> - 세 데이터셋의 구성을 비교해 보면 또 어떤 문제가 생길 수 있을까?
>
> 분석한 문제점을 해결할 방법을 제시하고, 이를 반영해 모델을 개선한 후 무엇이 좋아졌는지
> 확인해 보자.
>
> 힌트: 검증 정확도가 가질 수 있는 값을 모두 적어 보자.

In [14]:
# 1단계 - 데이터셋 구성 확인
from collections import Counter

print(f"{'데이터셋':<8}{'NORMAL':>10}{'PNEUMONIA':>12}{'합계':>8}{'폐렴 비율':>12}")
print('-' * 52)
splits = {}
for split, dataset in (('훈련', train_set), ('검증', valid_set), ('평가', test_set)):
    counts = Counter(label for _, label in dataset.samples)
    normal, pneumonia = counts[0], counts[1]
    total = normal + pneumonia
    splits[split] = (normal, pneumonia)
    print(f'{split:<8}{normal:>10,}{pneumonia:>12,}{total:>8,}'
          f'{pneumonia / total * 100:>11.1f}%')

데이터셋        NORMAL   PNEUMONIA      합계       폐렴 비율
----------------------------------------------------
훈련           1,341       3,875   5,216       74.3%
검증               8           8      16       50.0%
평가             234         390     624       62.5%


In [15]:
# 2단계 - 힌트가 가리키는 지점: 검증 정확도가 가질 수 있는 값
valid_size = len(valid_set)
possible = [k / valid_size * 100 for k in range(valid_size + 1)]
print(f'검증 데이터셋의 샘플 수: {valid_size}개')
print(f'검증 정확도가 가질 수 있는 값은 {len(possible)}가지뿐이다.')
print('  ' + ', '.join(f'{v:.2f}%' for v in possible))
print(f'\n눈금 간격: {100 / valid_size:.2f}%p')
print('즉 샘플 하나의 정답 여부가 검증 정확도를 '
      f'{100 / valid_size:.2f}%p나 움직인다.')

# 참고: 평가 데이터셋이라면 눈금이 얼마나 촘촘한가
print(f'\n(비교) 평가 데이터셋 {len(test_set)}개의 눈금 간격: '
      f'{100 / len(test_set):.2f}%p')

검증 데이터셋의 샘플 수: 16개
검증 정확도가 가질 수 있는 값은 17가지뿐이다.
  0.00%, 6.25%, 12.50%, 18.75%, 25.00%, 31.25%, 37.50%, 43.75%, 50.00%, 56.25%, 62.50%, 68.75%, 75.00%, 81.25%, 87.50%, 93.75%, 100.00%

눈금 간격: 6.25%p
즉 샘플 하나의 정답 여부가 검증 정확도를 6.25%p나 움직인다.

(비교) 평가 데이터셋 624개의 눈금 간격: 0.16%p


In [16]:
# 3단계 - 항상 폐렴이라고 답하는 모델의 성능 (기준선)
print('한쪽 클래스만 찍는 모델의 정확도')
for split, (normal, pneumonia) in splits.items():
    total = normal + pneumonia
    print(f'  {split:<6} 항상 PNEUMONIA: {pneumonia / total * 100:5.2f}%   '
          f'항상 NORMAL: {normal / total * 100:5.2f}%')
print()
print('검증 데이터셋은 8:8 균형이라 아무 쪽이나 찍어도 50.00%지만,')
print('평가 데이터셋은 항상 PNEUMONIA만 찍어도 62.50%가 나온다.')

한쪽 클래스만 찍는 모델의 정확도
  훈련     항상 PNEUMONIA: 74.29%   항상 NORMAL: 25.71%
  검증     항상 PNEUMONIA: 50.00%   항상 NORMAL: 50.00%
  평가     항상 PNEUMONIA: 62.50%   항상 NORMAL: 37.50%

검증 데이터셋은 8:8 균형이라 아무 쪽이나 찍어도 50.00%지만,
평가 데이터셋은 항상 PNEUMONIA만 찍어도 62.50%가 나온다.


In [17]:
# 4단계 - 개선: 훈련 데이터셋 일부를 떼어 클래스 비율을 맞춘 검증 데이터셋을 새로 만든다
import random

VALID_RATIO = 0.1


def stratified_split(dataset, ratio, seed=SEED):
    """클래스 비율을 유지한 채 인덱스를 두 묶음으로 나눈다(층화 분할)"""
    by_class = {}
    for index, (_, label) in enumerate(dataset.samples):
        by_class.setdefault(label, []).append(index)
    rng = random.Random(seed)
    held_out, kept = [], []
    for label, indices in sorted(by_class.items()):
        shuffled = indices[:]
        rng.shuffle(shuffled)
        cut = int(len(shuffled) * ratio)
        held_out.extend(shuffled[:cut])
        kept.extend(shuffled[cut:])
    return sorted(kept), sorted(held_out)


from torch.utils.data import Subset

train_index, valid_index = stratified_split(train_set, VALID_RATIO)
new_train_set = Subset(train_set, train_index)
new_valid_set = Subset(train_set, valid_index)

# 원래 검증 데이터셋 16개도 버리지 않고 새 검증 데이터셋에 합친다
from torch.utils.data import ConcatDataset

new_valid_set = ConcatDataset([new_valid_set, valid_set])

new_train_loader = DataLoader(new_train_set, batch_size=BATCH_SIZE, shuffle=True)
new_valid_loader = DataLoader(new_valid_set, batch_size=BATCH_SIZE, shuffle=False)


def count_labels(dataset):
    counter = Counter()
    for i in range(len(dataset)):
        counter[dataset[i][1]] += 1
    return counter


new_valid_counts = Counter()
for index in valid_index:
    new_valid_counts[train_set.samples[index][1]] += 1
new_valid_counts[0] += splits['검증'][0]
new_valid_counts[1] += splits['검증'][1]

print(f'새 훈련 데이터셋: {len(new_train_set):,}개')
print(f'새 검증 데이터셋: {len(new_valid_set):,}개 '
      f'(NORMAL {new_valid_counts[0]}, PNEUMONIA {new_valid_counts[1]}, '
      f'폐렴 비율 {new_valid_counts[1] / len(new_valid_set) * 100:.1f}%)')
print(f'검증 정확도 눈금 간격: {100 / len(new_valid_set):.3f}%p '
      f'(원래 {100 / valid_size:.2f}%p)')

새 훈련 데이터셋: 4,695개
새 검증 데이터셋: 537개 (NORMAL 142, PNEUMONIA 395, 폐렴 비율 73.6%)
검증 정확도 눈금 간격: 0.186%p (원래 6.25%p)


In [18]:
# 5단계 - 새 데이터셋 구성으로 다시 학습 ([연습 문제 8-14]와 같은 조건)
model_rs = build_finetune()
resplit_groups = [
    {'params': [p for name, p in model_rs.named_parameters()
                if not name.startswith('fc.')], 'lr': LR_FT},
    {'params': model_rs.fc.parameters(), 'lr': LR},
]

model_resplit, results['resplit'] = transfer_train(
    model_rs, new_train_loader, new_valid_loader, test_loader,
    '미세 조정(검증 재분할)', param_groups=resplit_groups)


미세 조정(검증 재분할) 학습 (학습 대상 파라미터 23,512,130개)


에포크    훈련 손실    검증 손실    정확도(%)     시간
  1/20       0.3252       0.2133       91.62%     1:22


  2/20       0.1757       0.1802       92.92%     2:42


  3/20       0.1447       0.1481       93.85%     4:02


  4/20       0.1187       0.1398       93.67%     5:22


...

 11/20       0.0425       0.1145       95.72%    14:41


 12/20       0.0370       0.1174       96.28%    16:01


 13/20       0.0335       0.1129       95.72%    17:21


 14/20       0.0266       0.1283       96.09%    18:41
최적 에포크(9, 최소 검증 손실 0.1116)의 파라미터로 복원


미세 조정(검증 재분할): 최적 에포크 9, 평가 정확도 81.73%


In [19]:
# 6단계 - 무엇이 좋아졌는지 확인
#     정확도만 보면 훈련 데이터가 줄어 오히려 나빠질 수도 있다.
#     이 문제의 개선 대상은 '조기 종료 판단의 신뢰성'이다.
print(f"{'방식':<26}{'최적 에포크':>10}{'평가 정확도':>14}")
print('-' * 50)
for k in ('finetune', 'resplit'):
    r = results[k]
    print(f"{r['name']:<26}{r['best_epoch']:>10}{r['test_acc']:>13.2f}%")

print()
print(f'검증 데이터셋 크기: {valid_size}개 -> {len(new_valid_set):,}개')
print(f'검증 정확도 눈금:   {100 / valid_size:.2f}%p -> '
      f'{100 / len(new_valid_set):.3f}%p')
print(f'검증 폐렴 비율:     {splits["검증"][1] / valid_size * 100:.1f}% -> '
      f'{new_valid_counts[1] / len(new_valid_set) * 100:.1f}% '
      f'(평가 데이터셋 {splits["평가"][1] / sum(splits["평가"]) * 100:.1f}%)')

방식                            최적 에포크        평가 정확도
--------------------------------------------------
미세 조정                              3        82.05%
미세 조정(검증 재분할)                      9        81.73%

검증 데이터셋 크기: 16개 -> 537개
검증 정확도 눈금:   6.25%p -> 0.186%p
검증 폐렴 비율:     50.0% -> 73.6% (평가 데이터셋 62.5%)


### 풀이 해설 — 연습 문제 8-15

**첫 번째 질문 — 검증 데이터셋으로 조기 종료 시점을 판단할 때의 문제**

검증 데이터셋은 16개(클래스마다 8개)뿐이다. 그래서 검증 정확도는 **17가지 값밖에 가질 수 없고,
눈금 간격이 6.25%p**다. 샘플 하나를 더 맞히느냐 마느냐가 정확도를 6.25%p나 움직인다. 본문
예제의 학습 로그에 75.00%, 81.25%, 87.50% 같은 값만 찍히는 것이 그 때문이다.

이렇게 거친 눈금으로는 **모델이 정말 좋아졌는지 우연히 한 장을 더 맞혔는지 구분할 수 없다.**
조기 종료는 검증 지표가 나아지지 않는 시점을 찾는 장치인데, 지표 자체가 요동치면 엉뚱한
에포크에서 멈추거나 반대로 멈춰야 할 때 멈추지 못한다.

**두 번째 질문 — 세 데이터셋의 구성을 비교했을 때의 문제**

| 데이터셋 | NORMAL | PNEUMONIA | 폐렴 비율 |
|---|---|---|---|
| 훈련 | 1,341 | 3,875 | 74.3% |
| 검증 | 8 | 8 | **50.0%** |
| 평가 | 234 | 390 | 62.5% |

**검증 데이터셋만 클래스가 균형을 이루고, 훈련과 평가는 폐렴 쪽으로 기울어 있다.** 세 데이터셋이
서로 다른 분포를 갖는 셈이다. 그 결과 검증 성능이 평가 성능을 제대로 대변하지 못한다.
예를 들어 훈련 데이터의 치우침 때문에 폐렴 쪽으로 기운 모델은 균형 잡힌 검증 데이터셋에서
불리하게 평가되지만, 정작 평가 데이터셋에서는 유리하다.

덧붙여, **평가 데이터셋에서 항상 폐렴이라고만 답해도 62.5%가 나온다.** 모델의 정확도를 이
기준선과 견주지 않으면 성능을 과대평가하기 쉽다.

**해결 방법과 개선**

훈련 데이터셋의 10%를 **클래스 비율을 유지한 채**(층화 분할) 떼어 새 검증 데이터셋을 만들고,
원래의 16개도 여기에 합쳤다.

| | 원래 | 재분할 후 |
|---|---|---|
| 검증 샘플 수 | 16개 | **537개** |
| 검증 정확도 눈금 | 6.25%p | **0.186%p** |
| 검증 폐렴 비율 | 50.0% | 73.6% |
| 최적 에포크 | 3 | **9** |
| 평가 정확도 | 82.05% | 81.73% |

**정확도는 0.32%p 떨어졌다.** 훈련 데이터의 10%를 검증으로 떼어 냈으니 당연하다.
**이 문제가 고친 것은 정확도가 아니다.**

**좋아진 것은 조기 종료 판단의 신뢰성이다.** 눈금이 6.25%p에서 0.186%p로 34배 촘촘해졌고,
최적 에포크가 3에서 9로 늘었다. 원래는 검증 지표가 거칠어 **우연한 한두 샘플의 정답 여부로
일찍 멈춰 버렸던** 것이다. 이제는 검증 손실이 실제로 나아지지 않는 시점까지 학습이 이어진다.

**그런데 한 가지는 해결되지 않았다.** 층화 분할은 **훈련 데이터의 분포를 물려받으므로**,
새 검증 데이터셋의 폐렴 비율은 73.6%로 훈련(74.3%)과는 잘 맞지만 **평가(62.5%)와는 여전히
어긋난다.** 검증 데이터셋이 대변해야 하는 것은 모델이 실제로 마주할 분포, 즉 평가 데이터셋
쪽이므로 이것은 남은 문제다.

평가 분포에 맞추려면 클래스별로 뽑는 비율을 다르게 해야 한다(정상을 더 많이, 폐렴을 덜).
다만 그러면 훈련 데이터의 정상 샘플이 더 줄어드는 대가를 치른다. **정답이 하나로 떨어지지
않는다는 것까지가 이 도전 문제의 수확이다.** 두 번째 물음("세 데이터셋의 구성을 비교해 보면
또 어떤 문제가 생길 수 있을까?")이 겨눈 문제는 검증 데이터셋을 다시 만드는 것만으로는
온전히 풀리지 않는다.

---

## 연습 문제 8-16 [도전 문제]

> [연습 문제 8-14]의 미세 조정 모델에 개선 아이디어를 하나 더 적용해 보자. ResNet-50의 첫 번째
> 합성곱 계층은 ImageNet에 맞춰 3채널 컬러 이미지를 입력받으므로, 회색조 흉부 X선 사진은 데이터
> 변환 객체가 R, G, B 세 채널로 복사해 넣고 있다. 이번에는 첫 번째 합성곱 계층이 1채널 이미지를
> 직접 입력받도록 모델을 손보는 쪽으로 바꿔 보자. 다음과 같은 절차로 진행하면 된다.
> - 데이터 변환 객체 수정: 채널 보정 없이 이미지를 ImageNet 학습 때의 입력 크기인 224x224로
>   조정하고, 평균 0.5, 표준편차 0.5로 정규화하는 변환 객체를 새로 만들어 사용한다.
> - 모델 구조 수정: 1채널 입력을 받도록 첫 번째 합성곱 계층을 교체한다. 앞에서 해 두지 않았다면
>   분류기 계층도 출력이 2가 되도록 교체한다. 필수는 아니지만 기존 첫 번째 합성곱 계층의 학습된
>   파라미터를 새 첫 번째 계층의 초깃값으로 이식하면 성능이 조금 나아질 수 있다. 세 채널의
>   가중치를 합산한 값을 써 보고, 왜 그것이 합리적인지 생각해 보자.
> - 전이 학습: 미세 조정 방식으로 모델을 학습한 후, 새로 교체한 두 계층만 학습하는 특징 추출
>   방식으로도 학습해 결과를 비교해 보자.

In [20]:
# 절차 1 - 1채널 데이터 변환 객체를 새로 만든다
from torchvision import transforms

gray_transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),   # 3채널 복사를 하지 않는다
    transforms.Resize((224, 224)),                 # ImageNet 학습 때의 입력 크기
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,)),
])

gray_train_set = ImageFolder(root=xray_root / 'train', transform=gray_transform)
gray_valid_set = ImageFolder(root=xray_root / 'val', transform=gray_transform)
gray_test_set = ImageFolder(root=xray_root / 'test', transform=gray_transform)

gray_train_loader = DataLoader(gray_train_set, batch_size=BATCH_SIZE, shuffle=True)
gray_valid_loader = DataLoader(gray_valid_set, batch_size=BATCH_SIZE, shuffle=False)
gray_test_loader = DataLoader(gray_test_set, batch_size=BATCH_SIZE, shuffle=False)

sample_tensor, _ = gray_train_set[0]
print(f'변환 후 샘플 형태: {tuple(sample_tensor.shape)}')
print(f'값의 범위: {sample_tensor.min():.3f} ~ {sample_tensor.max():.3f}')

변환 후 샘플 형태: (1, 224, 224)
값의 범위: -1.000 ~ 0.945


In [21]:
# 절차 2 - 첫 번째 합성곱 계층을 1채널 입력용으로 교체한다
def build_gray_model(transplant=True, finetune=True):
    """transplant: 세 채널 가중치를 합산해 새 계층의 초깃값으로 이식할지 여부
    finetune: True면 미세 조정, False면 교체한 두 계층만 학습하는 특징 추출"""
    common.set_seed(SEED, deterministic=True)
    model = timm.create_model('resnet50', pretrained=True)
    old_conv = model.conv1

    new_conv = nn.Conv2d(
        in_channels=1,
        out_channels=old_conv.out_channels,
        kernel_size=old_conv.kernel_size,
        stride=old_conv.stride,
        padding=old_conv.padding,
        bias=(old_conv.bias is not None),
    )
    if transplant:
        with torch.no_grad():
            # 세 채널의 가중치를 더해 1채널 가중치로 삼는다
            new_conv.weight.copy_(old_conv.weight.sum(dim=1, keepdim=True))
            if old_conv.bias is not None:
                new_conv.bias.copy_(old_conv.bias)

    if not finetune:
        # 특징 추출 방식: 먼저 전체를 고정한 뒤 새로 넣을 두 계층만 남긴다
        for param in model.parameters():
            param.requires_grad = False

    model.conv1 = new_conv
    model.fc = nn.Linear(2048, 2)
    return model


for finetune in (True, False):
    m = build_gray_model(finetune=finetune)
    trainable = sum(p.numel() for p in m.parameters() if p.requires_grad)
    label = '미세 조정' if finetune else '특징 추출'
    print(f'{label}: 학습 대상 파라미터 {trainable:,}개')

미세 조정: 학습 대상 파라미터 23,505,858개


특징 추출: 학습 대상 파라미터 7,234개


In [22]:
# 가중치 합산이 왜 합리적인가 - 숫자로 확인
check = timm.create_model('resnet50', pretrained=True)
old_w = check.conv1.weight.detach()          # (64, 3, 7, 7)
summed = old_w.sum(dim=1, keepdim=True)      # (64, 1, 7, 7)

# 회색조 이미지를 3채널로 복사한 입력
gray_image = torch.rand(1, 1, 32, 32)
rgb_copy = gray_image.repeat(1, 3, 1, 1)

import torch.nn.functional as F

out_rgb = F.conv2d(rgb_copy, old_w, stride=2, padding=3)
out_gray_sum = F.conv2d(gray_image, summed, stride=2, padding=3)
out_gray_mean = F.conv2d(gray_image, old_w.mean(dim=1, keepdim=True),
                         stride=2, padding=3)

print(f'3채널 복사 입력의 출력과 비교한 최대 오차')
print(f'  세 채널 가중치를 합산:  {(out_rgb - out_gray_sum).abs().max():.3e}')
print(f'  세 채널 가중치를 평균:  {(out_rgb - out_gray_mean).abs().max():.3e}')
print()
print('합산은 출력을 그대로 보존하고, 평균은 값을 1/3로 줄인다.')
print(f'  합산 출력의 표준편차: {out_gray_sum.std():.4f}')
print(f'  평균 출력의 표준편차: {out_gray_mean.std():.4f}')

3채널 복사 입력의 출력과 비교한 최대 오차
  세 채널 가중치를 합산:  8.583e-06
  세 채널 가중치를 평균:  1.099e+01

합산은 출력을 그대로 보존하고, 평균은 값을 1/3로 줄인다.
  합산 출력의 표준편차: 2.6931
  평균 출력의 표준편차: 0.8977


In [23]:
# 1채널 미세 조정
#     conv1과 fc는 새로 만든 계층이므로 통상 학습률,
#     나머지 사전 학습된 계층은 낮은 학습률을 적용한다
model_gft = build_gray_model(finetune=True)
gray_groups = [
    {'params': [p for name, p in model_gft.named_parameters()
                if not (name.startswith('conv1.') or name.startswith('fc.'))],
     'lr': LR_FT},
    {'params': list(model_gft.conv1.parameters())
               + list(model_gft.fc.parameters()), 'lr': LR},
]

model_gray_ft, results['gray_finetune'] = transfer_train(
    model_gft, gray_train_loader, gray_valid_loader, gray_test_loader,
    '1채널 미세 조정', param_groups=gray_groups)


1채널 미세 조정 학습 (학습 대상 파라미터 23,505,858개)


에포크    훈련 손실    검증 손실    정확도(%)     시간
  1/20       0.3063       0.8961       56.25%     0:57


  2/20       0.1589       0.7533       68.75%     1:59


  3/20       0.1272       0.4432       81.25%     3:02


  4/20       0.1074       0.4760       68.75%     4:02


...

 15/20       0.0357       0.6064       81.25%    14:59


 16/20       0.0341       0.3785       87.50%    16:10


 17/20       0.0251       0.3662       87.50%    17:09


 18/20       0.0210       0.4652       81.25%    18:30
최적 에포크(13, 최소 검증 손실 0.3300)의 파라미터로 복원


1채널 미세 조정: 최적 에포크 13, 평가 정확도 84.78%


In [24]:
model_gray_fe, results['gray_feature'] = transfer_train(
    build_gray_model(finetune=False), gray_train_loader, gray_valid_loader,
    gray_test_loader, '1채널 특징 추출', lr=1e-3)

1채널 특징 추출 학습 (학습 대상 파라미터 7,234개)


에포크    훈련 손실    검증 손실    정확도(%)     시간
  1/20       0.3432       0.7221       56.25%     1:05


  2/20       0.2056       0.7906       68.75%     2:04


  3/20       0.1759       0.6368       68.75%     3:07


  4/20       0.1593       0.4535       75.00%     4:01


...

  6/20       0.1442       0.4648       68.75%     5:51


  7/20       0.1370       0.8303       56.25%     6:50


  8/20       0.1297       0.6356       68.75%     7:58


  9/20       0.1189       0.5336       68.75%     9:25
최적 에포크(4, 최소 검증 손실 0.4535)의 파라미터로 복원


1채널 특징 추출: 최적 에포크 4, 평가 정확도 82.53%


In [25]:
print(f"{'방식':<26}{'최적 에포크':>10}{'평가 정확도':>14}")
print('-' * 50)
for k in ('replaced', 'finetune', 'gray_finetune', 'gray_feature'):
    r = results[k]
    print(f"{r['name']:<26}{r['best_epoch']:>10}{r['test_acc']:>13.2f}%")

방식                            최적 에포크        평가 정확도
--------------------------------------------------
교체 방식(특징 추출)                       6        80.29%
미세 조정                              3        82.05%
1채널 미세 조정                         13        84.78%
1채널 특징 추출                          4        82.53%


### 풀이 해설 — 연습 문제 8-16

**왜 세 채널의 가중치를 합산하는 것이 합리적인가**

지금까지는 회색조 X선 사진을 R, G, B 세 채널에 **똑같이 복사해** 넣었다. 즉 세 채널의 값이
모두 같다. 합성곱의 출력은 채널별 가중치와 입력의 곱을 모두 더한 값이므로, 입력값을 `v`라 하면

```
출력 = W_R * v + W_G * v + W_B * v = (W_R + W_G + W_B) * v
```

가 된다. **세 채널의 가중치를 더한 것이 곧 1채널 가중치다.** 위 셀에서 확인했듯 합산한 가중치로
1채널 입력을 통과시킨 결과는 3채널 복사 입력의 결과와 **오차가 사실상 0**이다.

평균을 쓰면 값이 1/3로 줄어들어 뒤따르는 배치 정규화 계층이 보던 분포가 달라진다. 사전 학습된
이동평균과 이동분산이 어긋나므로 학습 초반이 불안정해진다. **합산은 사전 학습 상태를 그대로
이어받는 유일한 선택이다.**

**두 학습 방식의 비교**

특징 추출 방식에서 학습하는 계층은 `conv1`과 `fc` **두 개**다. 둘 다 새로 만든 계층이라 사전
학습된 파라미터가 없기 때문이다(가중치를 이식했더라도 이후 학습 대상으로 남긴다). 나머지 합성곱
계층은 모두 고정된다.

주의할 점은 **가중치 이식이 특징 추출 방식에서 더 중요하다**는 것이다. 미세 조정은 모델 전체가
함께 움직이므로 첫 계층이 무작위로 시작해도 따라잡을 여지가 있지만, 특징 추출은 뒤쪽 계층이
모두 고정되어 있어 첫 계층이 엉뚱한 특징을 내보내면 회복할 방법이 없다.

실행 결과는 위 표로 확인한다. `[연습 문제 8-14]`의 3채널 미세 조정과 견주면, 채널 보정을
없앤 것이 실제로 도움이 되었는지 알 수 있다.